# Brain Tumor Modeling and Training

This notebook walks through the full training workflow step by step.

Goals:
- Train a custom CNN baseline.
- Train an EfficientNet transfer learning model.
- Evaluate with accuracy, precision, recall, F1-score, specificity, confusion matrix, and ROC-AUC.
- Save artifacts so they can be reused in the explainability notebook and Streamlit dashboard.

Literature used as guidance:
- Frontiers 2024: custom CNNs can perform strongly for multiclass brain MRI classification when training and evaluation are handled carefully.
- Springer 2025: structured preprocessing, model comparison, and transparent reporting are important, but overlap metrics like Dice and IoU require real tumor masks.

In [ ]:
# Setup: import libraries and define project-level constants.
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Fix the random seed so notebook runs are easier to reproduce.
tf.keras.utils.set_random_seed(42)

PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / 'Dataset' / 'split_data'
RESULTS_ROOT = PROJECT_ROOT / 'results'
MODELS_ROOT = PROJECT_ROOT / 'artifacts' / 'models'
CLASSES = ['glioma', 'healthy', 'meningioma', 'pituitary']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 20
FINE_TUNE_EPOCHS = 8

# Create output folders once so later cells can save models and plots safely.
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Results root:', RESULTS_ROOT)


## Step 1: Confirm the Dataset Structure

Before training, we make sure the split dataset exists in the expected structure:
`Dataset/split_data/train|val|test/<class_name>`.

This step prevents downstream errors such as empty datasets, missing class folders, or incorrect file paths.

In [ ]:
# Step 1: validate the split dataset structure and summarize class counts.
def check_dataset_structure(dataset_root, class_names):
    # Collect missing folders first so the error message is easy to understand.
    missing_paths = []
    for split_name in ['train', 'val', 'test']:
        split_dir = dataset_root / split_name
        if not split_dir.exists():
            missing_paths.append(split_dir)
            continue
        for class_name in class_names:
            class_dir = split_dir / class_name
            if not class_dir.exists():
                missing_paths.append(class_dir)
    if missing_paths:
        raise FileNotFoundError('Missing dataset paths:\n' + '\n'.join(str(path) for path in missing_paths))


def build_split_summary(dataset_root, class_names):
    # Count image files per split and class for a quick balance check.
    rows = []
    valid_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    for split_name in ['train', 'val', 'test']:
        for class_name in class_names:
            class_dir = dataset_root / split_name / class_name
            count = 0
            if class_dir.exists():
                count = sum(1 for file in class_dir.iterdir() if file.is_file() and file.suffix.lower() in valid_suffixes)
            rows.append({'split': split_name, 'class_name': class_name, 'image_count': count})
    return pd.DataFrame(rows)


check_dataset_structure(DATASET_ROOT, CLASSES)
split_summary = build_split_summary(DATASET_ROOT, CLASSES)
display(split_summary.pivot(index='class_name', columns='split', values='image_count').fillna(0).astype(int))

## Step 2: Build TensorFlow Datasets

We now load the train, validation, and test folders into TensorFlow datasets. Keeping this step explicit in the notebook makes it easier to inspect batch sizes, image resizing, and class ordering.

In [ ]:
# Step 2: build TensorFlow datasets and define augmentation.
# Use moderate augmentation that keeps MRI anatomy plausible.
augmentation_layer = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ],
    name='augmentation_layer',
)


def build_dataset(split_name, shuffle):
    # Load images directly from the directory structure inferred from class folders.
    return tf.keras.utils.image_dataset_from_directory(
        DATASET_ROOT / split_name,
        labels='inferred',
        label_mode='categorical',
        class_names=CLASSES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )


train_ds = build_dataset('train', shuffle=True)
val_ds = build_dataset('val', shuffle=False)
test_ds = build_dataset('test', shuffle=False)

# Prefetch batches so training can overlap data preparation and GPU/CPU work.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())
print('Validation batches:', tf.data.experimental.cardinality(val_ds).numpy())
print('Test batches:', tf.data.experimental.cardinality(test_ds).numpy())

## Step 3: Visualize Augmentation

This preview checks whether augmentation is helping rather than distorting the MRI slices. If images become too dark or unrealistic, this is the place to catch it before training.

In [ ]:
# Step 3: preview original versus augmented MRI images.
for images, labels in train_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i in range(4):
        # Show the raw resized image.
        original = images[i].numpy().astype(np.uint8)
        # Apply augmentation in training mode so random transforms are visible.
        augmented = augmentation_layer(tf.expand_dims(images[i], axis=0), training=True)[0].numpy()
        augmented = np.clip(augmented, 0, 255).astype(np.uint8)

        axes[0, i].imshow(original)
        axes[0, i].set_title(f'Original\n{CLASSES[np.argmax(labels[i].numpy())]}')
        axes[0, i].axis('off')

        axes[1, i].imshow(augmented)
        axes[1, i].set_title('Augmented')
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()

## Helper Functions

The next cell defines the reusable helper functions for model compilation, training callbacks, evaluation, specificity, confusion matrix creation, and ROC plotting. Keeping them in one place makes the later training sections easier to read.

In [ ]:
# Helper section: define training, evaluation, and plotting utilities.
def compile_model(model, learning_rate):
    # Compile once here so both models use the same loss and metrics.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc', multi_label=True, num_labels=len(CLASSES)),
        ],
    )
    return model


def build_custom_cnn():
    # A straightforward CNN baseline inspired by common medical-image classifiers.
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = augmentation_layer(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    for filters in [32, 64, 128, 256]:
        # Two convolutions per block let the model learn richer local patterns.
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name='custom_cnn')
    return compile_model(model, 1e-3)


def build_efficientnet_model():
    # Transfer learning usually gives a stronger starting point than training from scratch.
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = augmentation_layer(inputs)
    x = tf.keras.applications.efficientnet.preprocess_input(x)

    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMAGE_SIZE, 3),
    )
    # Freeze the pretrained backbone for the first training phase.
    base_model.trainable = False

    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name='efficientnet_b0')
    model = compile_model(model, 1e-3)
    return model, base_model


def get_callbacks(model_name):
    # Save the best checkpoint and stop early when validation stops improving.
    model_dir = RESULTS_ROOT / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(model_dir / 'best_model.keras', monitor='val_loss', save_best_only=True),
    ]


def collect_predictions(model, dataset):
    # Predict probabilities first, then derive the predicted class for each sample.
    probabilities = model.predict(dataset, verbose=0)
    y_pred = probabilities.argmax(axis=1)

    label_batches = []
    for _, batch_labels in dataset:
        label_batches.append(batch_labels.numpy())
    y_true = np.concatenate(label_batches).argmax(axis=1)
    return y_true, y_pred, probabilities


def compute_specificity(conf_matrix):
    # Specificity is computed one-vs-rest for each class.
    scores = {}
    total = conf_matrix.sum()
    for i, class_name in enumerate(CLASSES):
        tp = conf_matrix[i, i]
        fp = conf_matrix[:, i].sum() - tp
        fn = conf_matrix[i, :].sum() - tp
        tn = total - tp - fp - fn
        denominator = tn + fp
        scores[class_name] = float(tn / denominator) if denominator else 0.0
    return scores


def evaluate_model(model, dataset, model_name):
    # Gather prediction outputs, then generate all core evaluation artifacts.
    y_true, y_pred, probabilities = collect_predictions(model, dataset)
    report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
    conf_matrix = confusion_matrix(y_true, y_pred)
    specificity = compute_specificity(conf_matrix)

    metrics = {
        'accuracy': float(report['accuracy']),
        'macro_avg': {
            'precision': float(report['macro avg']['precision']),
            'recall': float(report['macro avg']['recall']),
            'f1_score': float(report['macro avg']['f1-score']),
            'roc_auc_ovr': float(roc_auc_score(tf.keras.utils.to_categorical(y_true, num_classes=len(CLASSES)), probabilities, multi_class='ovr', average='macro')),
        },
        'per_class': {},
        'confusion_matrix': conf_matrix.tolist(),
        'class_names': CLASSES,
    }

    for idx, class_name in enumerate(CLASSES):
        metrics['per_class'][class_name] = {
            'precision': float(report[class_name]['precision']),
            'recall': float(report[class_name]['recall']),
            'f1_score': float(report[class_name]['f1-score']),
            'support': float(report[class_name]['support']),
            'specificity': float(specificity[class_name]),
            'roc_auc_ovr': float(roc_auc_score((y_true == idx).astype(int), probabilities[:, idx])),
        }

    output_dir = RESULTS_ROOT / model_name
    output_dir.mkdir(parents=True, exist_ok=True)

    with open(output_dir / 'metrics.json', 'w') as file:
        json.dump(metrics, file, indent=2)

    # Save a confusion matrix image for the notebook and dashboard.
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(output_dir / 'confusion_matrix.png', dpi=150)
    plt.show()

    # Plot one-vs-rest ROC curves for each class.
    plt.figure(figsize=(8, 6))
    y_true_one_hot = tf.keras.utils.to_categorical(y_true, num_classes=len(CLASSES))
    for idx, class_name in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(y_true_one_hot[:, idx], probabilities[:, idx])
        plt.plot(fpr, tpr, label=f"{class_name} (AUC={metrics['per_class'][class_name]['roc_auc_ovr']:.3f})")
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.title(f'{model_name} ROC Curves')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(output_dir / 'roc_curves.png', dpi=150)
    plt.show()

    return metrics


def plot_history(history, model_name):
    # Plot training and validation trends to check for overfitting.
    output_dir = RESULTS_ROOT / model_name
    output_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='train')
    axes[0].plot(history.history['val_accuracy'], label='val')
    axes[0].set_title(f'{model_name} Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='train')
    axes[1].plot(history.history['val_loss'], label='val')
    axes[1].set_title(f'{model_name} Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(output_dir / 'training_history.png', dpi=150)
    plt.show()

## Optional: Load Saved Results Instead of Retraining

If the notebook was reloaded and the visible outputs disappeared, use the next cells to recover the saved metrics, plots, and trained models from disk without rerunning the full training process.

### Code Cell: Load Saved Metrics and Plots

This cell reloads saved metrics from JSON files and displays plots that were already generated during training.

In [ ]:
# Optional recovery step: load saved metrics and plots without retraining.
from IPython.display import Image as IPythonImage, display


def load_saved_metrics(model_name):
    metrics_path = RESULTS_ROOT / model_name / 'metrics.json'
    if not metrics_path.exists():
        print(f'No saved metrics found for {model_name}.')
        return None
    with open(metrics_path, 'r') as file:
        return json.load(file)


def show_saved_artifacts(model_name):
    output_dir = RESULTS_ROOT / model_name
    for filename in ['training_history.png', 'confusion_matrix.png', 'roc_curves.png']:
        artifact_path = output_dir / filename
        if artifact_path.exists():
            print(f'{model_name}: {filename}')
            display(IPythonImage(filename=str(artifact_path)))
        else:
            print(f'{model_name}: missing {filename}')


saved_cnn_metrics = load_saved_metrics('custom_cnn')
saved_efficientnet_metrics = load_saved_metrics('efficientnet_b0')

if saved_cnn_metrics is not None:
    print('Custom CNN metrics loaded successfully.')
    display(pd.DataFrame(saved_cnn_metrics['per_class']).T)
    show_saved_artifacts('custom_cnn')

if saved_efficientnet_metrics is not None:
    print('EfficientNetB0 metrics loaded successfully.')
    display(pd.DataFrame(saved_efficientnet_metrics['per_class']).T)
    show_saved_artifacts('efficientnet_b0')

### Code Cell: Reload Saved Models

Run this if you want to continue working with the trained models in later cells without repeating the full training run.

In [ ]:
# Optional recovery step: reload the saved Keras model files.
saved_custom_cnn_path = MODELS_ROOT / 'custom_cnn.keras'
saved_efficientnet_path = MODELS_ROOT / 'efficientnet_b0.keras'

reloaded_cnn_model = tf.keras.models.load_model(saved_custom_cnn_path) if saved_custom_cnn_path.exists() else None
reloaded_efficientnet_model = tf.keras.models.load_model(saved_efficientnet_path) if saved_efficientnet_path.exists() else None

print('Reloaded custom CNN:', reloaded_cnn_model is not None)
print('Reloaded EfficientNetB0:', reloaded_efficientnet_model is not None)

## Step 4: Train the Custom CNN

This is the baseline model. It helps us answer whether a simpler architecture is already strong enough before we move to transfer learning.

## Train Custom CNN

In [ ]:
# Step 4: initialize the custom CNN baseline.
cnn_model = build_custom_cnn()
cnn_model.summary()

In [ ]:
# Step 4: train and evaluate the custom CNN baseline.
cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks('custom_cnn'),
    verbose=1,
)

cnn_model.save(MODELS_ROOT / 'custom_cnn.keras')
plot_history(cnn_history, 'custom_cnn')
cnn_metrics = evaluate_model(cnn_model, test_ds, 'custom_cnn')
pd.DataFrame(cnn_metrics['per_class']).T

## Step 5: Train EfficientNetB0

This model uses transfer learning. We first train only the classification head, then unfreeze the deeper layers for fine-tuning with a smaller learning rate.

In [ ]:
# Step 5: initialize the EfficientNetB0 transfer-learning model.
efficientnet_model, efficientnet_backbone = build_efficientnet_model()
efficientnet_model.summary()

In [ ]:
# Step 5: train EfficientNetB0 in two stages and then evaluate it.
# Stage 1: train only the top classification layers.
efficientnet_history_stage_1 = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks('efficientnet_b0'),
    verbose=1,
)

# Stage 2: unfreeze part of the backbone for fine-tuning.
efficientnet_backbone.trainable = True
for layer in efficientnet_backbone.layers[:180]:
    layer.trainable = False

efficientnet_model = compile_model(efficientnet_model, 1e-5)

efficientnet_history_stage_2 = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=get_callbacks('efficientnet_b0'),
    verbose=1,
)

# Merge the two training histories so we can plot one continuous curve.
combined_history = type('HistoryContainer', (), {})()
combined_history.history = {
    key: efficientnet_history_stage_1.history.get(key, []) + efficientnet_history_stage_2.history.get(key, [])
    for key in set(efficientnet_history_stage_1.history) | set(efficientnet_history_stage_2.history)
}

efficientnet_model.save(MODELS_ROOT / 'efficientnet_b0.keras')
plot_history(combined_history, 'efficientnet_b0')
efficientnet_metrics = evaluate_model(efficientnet_model, test_ds, 'efficientnet_b0')
pd.DataFrame(efficientnet_metrics['per_class']).T

In [ ]:
# Step 6: compare both trained models side by side.
comparison = pd.DataFrame(
    {
        'custom_cnn': {
            'accuracy': cnn_metrics['accuracy'],
            'macro_f1': cnn_metrics['macro_avg']['f1_score'],
            'macro_roc_auc_ovr': cnn_metrics['macro_avg']['roc_auc_ovr'],
        },
        'efficientnet_b0': {
            'accuracy': efficientnet_metrics['accuracy'],
            'macro_f1': efficientnet_metrics['macro_avg']['f1_score'],
            'macro_roc_auc_ovr': efficientnet_metrics['macro_avg']['roc_auc_ovr'],
        },
    }
)

display(comparison)

with open(RESULTS_ROOT / 'model_comparison.json', 'w') as file:
    json.dump(comparison.to_dict(), file, indent=2)

best_model_name = comparison.loc['macro_f1'].astype(float).idxmax()
print('Best model based on macro F1:', best_model_name)

## Important Note About Dice and IoU

The current dataset is a classification dataset, not a segmentation dataset. That means Dice and IoU should only be reported if you later add expert tumor masks. The explainability notebook includes an optional section for that case.